### #6

Kaggle competition: [\[link\]](https://www.kaggle.com/competitions/playground-series-s5e6)

Entry by Robin R.P.M. Kras

### ⭐ 1. Introduction & Overview


Your Goal: Your task is to predict the top 3 best fertilizers for a certain soil setting.

### 🔹 2. Import Libraries & Set Up


In [1]:
# General
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Machine Learning
import xgboost as xg

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error, roc_auc_score
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

from imblearn.over_sampling import SMOTE

# Feature Importance & Explainability
import shap

# Settings
import warnings
warnings.filterwarnings("ignore")

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)

print("Libraries loaded. Ready to go!")

Libraries loaded. Ready to go!


### 🔹 3. Load & Explore Data


In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f"Train shape: {train.shape}, Test shape: {test.shape}")

Train shape: (750000, 10), Test shape: (250000, 9)


In [3]:
train.head()

,id,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,0,37,70,36,Clayey,Sugarcane,36,4,5,28-28
1,1,27,69,65,Sandy,Millets,30,6,18,28-28
2,2,29,63,32,Sandy,Millets,24,12,16,17-17-17
3,3,35,62,54,Sandy,Barley,39,12,4,10-26-26
4,4,35,58,43,Red,Paddy,37,2,16,DAP


In [4]:
fertilizers = train['Fertilizer Name'].unique()
for i in fertilizers:
    print(i)

28-28
17-17-17
10-26-26
DAP
20-20
14-35-14
Urea


In [5]:
train.isnull().sum()

id                 0
Temparature        0
Humidity           0
Moisture           0
Soil Type          0
Crop Type          0
Nitrogen           0
Potassium          0
Phosphorous        0
Fertilizer Name    0
dtype: int64

In [6]:
test.isnull().sum()

id             0
Temparature    0
Humidity       0
Moisture       0
Soil Type      0
Crop Type      0
Nitrogen       0
Potassium      0
Phosphorous    0
dtype: int64

In [7]:
train.columns = train.columns.str.replace(' ', '_')
test.columns = test.columns.str.replace(' ', '_')

In [8]:
CATS = []
NUMS = []

In [9]:
FEATURES = []

In [10]:
for col in train.columns:
    FEATURES.append(col)

In [11]:
for c in FEATURES:
    if train[c].dtype == "object":
        CATS.append(c)

for c in FEATURES:
    if c not in CATS:
        NUMS.append(c)

### 🔹 4. Data Visualization & EDA



In [12]:
def plot_nums(dataframe):
    float_cols = [col for col in dataframe.columns if dataframe[col].dtype == "float64" or dataframe[col].dtype == "int64"]

    cols_per_row = 3
    num_plots = len(float_cols)
    rows = (num_plots // cols_per_row) + (num_plots % cols_per_row > 0) 

    fig, axes = plt.subplots(rows, cols_per_row, figsize=(15, 5 * rows)) 
    axes = axes.flatten()  

    for idx, col in enumerate(float_cols):
        sns.histplot(dataframe[col], bins=50, kde=True, ax=axes[idx])
        axes[idx].set_title(f"Distribution of {col}")

    for i in range(idx + 1, len(axes)):
        fig.delaxes(axes[i])

    plt.tight_layout()
    plt.show()

In [13]:
if False:
    plot_nums(train.drop(columns=['id', 'Fertilizer_Name']))

### 🔹 5. Feature Engineering

In [14]:
train.dtypes

id                  int64
Temparature         int64
Humidity            int64
Moisture            int64
Soil_Type          object
Crop_Type          object
Nitrogen            int64
Potassium           int64
Phosphorous         int64
Fertilizer_Name    object
dtype: object

In [15]:
# train['Climate'] = (train['Humidity'] + train['Temparature'] + train['Moisture']) / 3
# test['Climate'] = (test['Humidity'] + test['Temparature'] + test['Moisture']) / 3

In [16]:
le = LabelEncoder()

CATS.append('Fertilizer_Name')

for col in CATS:
    train[col] = le.fit_transform(train[col])
    if col in test.columns:
        test[col] = le.fit_transform(test[col])

In [17]:
train.head()

,id,Temparature,Humidity,Moisture,Soil_Type,Crop_Type,Nitrogen,Potassium,Phosphorous,Fertilizer_Name
0,0,37,70,36,1,8,36,4,5,4
1,1,27,69,65,4,4,30,6,18,4
2,2,29,63,32,4,4,24,12,16,2
3,3,35,62,54,4,0,39,12,4,0
4,4,35,58,43,3,6,37,2,16,5


### 🔹 6. XGBoost, KFold, CV

In [18]:
from lightgbm import LGBMClassifier

In [19]:
X = train.drop(columns=['id', 'Fertilizer_Name'], errors='ignore')
X_test = test.drop(columns=['id'], errors='ignore')

y = train["Fertilizer_Name"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, shuffle=True, random_state=SEED)

def quick_eval(model, X_train, y_train):
    model.fit(X_train, y_train)

    predictions_val = model.predict(X_val)

    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

    cv_scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='precision')

    rmse = root_mean_squared_error(y_val, predictions_val)
    print(f"*** {model.__class__.__name__} ***")
    print(f"Root mean squared error (val): {rmse}")
    print(f"Mean CV MSLE: {-np.mean(cv_scores):.4f}")
    print(f"CV Std Dev: {np.std(cv_scores):.4f}")

    stars = len(model.__class__.__name__) + 8
    print("*" * stars)
    print("\n")

# quick_eval(xg.XGBClassifier(), X_train, y_train)
# quick_eval(LogisticRegression(), X_train, y_train)
# quick_eval(LGBMClassifier(), X_train, y_train)

if False:
    model.fit(X, y) 
    predictions = model.predict(X_test)

    sub1 = pd.read_csv("sample_submission.csv")

    sub1.Listening_Time_minutes = predictions 

    sub1.to_csv("xgboost.csv", index=False)

    print("Sub shape:", sub1.shape)
    sub1.head()

In [20]:
def mapk(actual, predicted, k=3):
    """
    Computes the mean average precision at k.
    actual: array-like of true labels
    predicted: array-like of lists of predicted labels (top k)
    """
    score = 0.0
    for a, p in zip(actual, predicted):
        try:
            score += 1.0 / (p.index(a) + 1) if a in p else 0.0
        except ValueError:
            score += 0.0
    return score / len(actual)

In [21]:
train.head()

,id,Temparature,Humidity,Moisture,Soil_Type,Crop_Type,Nitrogen,Potassium,Phosphorous,Fertilizer_Name
0,0,37,70,36,1,8,36,4,5,4
1,1,27,69,65,4,4,30,6,18,4
2,2,29,63,32,4,4,24,12,16,2
3,3,35,62,54,4,0,39,12,4,0
4,4,35,58,43,3,6,37,2,16,5


Experimenting

In [22]:
if False:
    ###
    soil_moisture_mean = train.groupby('Soil_Type')['Moisture'].transform('mean')
    train['Moisture_Deviation'] = train['Moisture'] - soil_moisture_mean

    soil_moisture_mean_test = test.groupby('Soil_Type')['Moisture'].transform('mean')
    test['Moisture_Deviation'] = test['Moisture'] - soil_moisture_mean_test
    ###

    train['Climate_Index'] = (
        0.3 * train['Temparature'] +
        0.1 * train['Humidity'] +
        0.6 * train['Moisture']
    )

    test['Climate_Index'] = (
        0.3 * test['Temparature'] +
        0.1 * test['Humidity'] +
        0.6 * test['Moisture']
    )

    train['Nitrogen_Sq'] = train['Nitrogen'] ** 2
    train['Temp_Moisture'] = train['Temparature'] * train['Moisture']

    test['Nitrogen_Sq'] = test['Nitrogen'] ** 2
    test['Temp_Moisture'] = test['Temparature'] * test['Moisture']

    train['N_K_ratio'] = train['Nitrogen'] / (train['Potassium'] + 1)
    train['N_P_ratio'] = train['Nitrogen'] / (train['Phosphorous'] + 1)
    train['K_P_ratio'] = train['Potassium'] / (train['Phosphorous'] + 1)

    test['N_K_ratio'] = test['Nitrogen'] / (test['Potassium'] + 1)
    test['N_P_ratio'] = test['Nitrogen'] / (test['Phosphorous'] + 1)
    test['K_P_ratio'] = test['Potassium'] / (test['Phosphorous'] + 1)

In [23]:
X = train.drop(columns=['id', 'Fertilizer_Name'], errors='ignore')
X_test = test.drop(columns=['id'], errors='ignore')

y = train["Fertilizer_Name"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, shuffle=True, random_state=SEED)

In [24]:
X_train.head()

,Temparature,Humidity,Moisture,Soil_Type,Crop_Type,Nitrogen,Potassium,Phosphorous
453635,28,51,47,2,2,20,17,24
11651,33,62,30,4,0,7,0,6
431999,38,59,41,2,6,24,11,42
529211,26,52,57,2,10,27,17,19
110925,37,61,35,0,6,25,14,16


In [26]:
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import make_scorer

def mapk_eval(y_true, y_pred_proba, k=3):
    top_k = np.argsort(y_pred_proba, axis=1)[:, -k:][:, ::-1]
    return mapk(y_true, top_k.tolist(), k=k)

def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 100, 400),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
        'random_state': SEED,
        'use_label_encoder': False,
        'eval_metric': 'mlogloss'
    }
    model = xg.XGBClassifier(**params)
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_val)
    score = mapk_eval(y_val.values, y_pred_proba, k=3)
    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("Best MAP@3:", study.best_value)
print("Best params:", study.best_params)

[I 2025-06-11 17:41:09,298] A new study created in memory with name: no-name-dd5b1a35-a095-4ac2-a92c-8fe87d4dee67
[I 2025-06-11 17:41:36,577] Trial 0 finished with value: 0.3231222222222909 and parameters: {'max_depth': 8, 'learning_rate': 0.06326287765150974, 'n_estimators': 162, 'subsample': 0.6821233517625884, 'colsample_bytree': 0.9638359504388128, 'gamma': 2.4792001687350584, 'reg_alpha': 1.0189751290331739, 'reg_lambda': 4.768573075328976}. Best is trial 0 with value: 0.3231222222222909.
[I 2025-06-11 17:41:55,650] Trial 1 finished with value: 0.31536555555561735 and parameters: {'max_depth': 6, 'learning_rate': 0.16841103012214031, 'n_estimators': 129, 'subsample': 0.8081429739265287, 'colsample_bytree': 0.8205096527873423, 'gamma': 3.3963004754342645, 'reg_alpha': 4.3330543885150945, 'reg_lambda': 2.569170589281376}. Best is trial 0 with value: 0.3231222222222909.
[I 2025-06-11 17:42:17,485] Trial 2 finished with value: 0.32085333333340355 and parameters: {'max_depth': 6, 'lear

Best MAP@3: 0.33531888888896916
Best params: {'max_depth': 10, 'learning_rate': 0.08058832222297987, 'n_estimators': 268, 'subsample': 0.8878145949016356, 'colsample_bytree': 0.7945209987411344, 'gamma': 0.6025081681020137, 'reg_alpha': 3.1628674276138575, 'reg_lambda': 0.5475825279790727}


`Best MAP@3: 0.33507222222230404`, optuna, XGBClassifier, extra features: moisture_deviation

`Best MAP@3: 0.3415511111111975`, optuna, XGBClassifier, no extra features

In [27]:
og_train = pd.read_csv('train.csv')
og_test = pd.read_csv('test.csv')

In [28]:
fertilizer_encoder = LabelEncoder()
original_fertilizer_names = og_train['Fertilizer Name'].unique()
fertilizer_encoder.fit(original_fertilizer_names)

LabelEncoder()

In [29]:
submission1 = pd.read_csv("sample_submission.csv")

model = xg.XGBClassifier(**study.best_params)
model.fit(X, y)

probs = model.predict_proba(X_test)
top3_indices = np.argsort(probs, axis=1)[:, -3:][:, ::-1]

top3_fertilizers = [' '.join(fertilizer_encoder.inverse_transform(row)) for row in top3_indices]

submission = pd.DataFrame({
    'ID': test['id'],
    'Fertilizer_Name': top3_fertilizers
})

submission.to_csv('submission500.csv', index=False)

In [30]:
submission

,ID,Fertilizer_Name
0,750000,DAP 10-26-26 28-28
1,750001,17-17-17 20-20 10-26-26
2,750002,20-20 28-28 10-26-26
3,750003,14-35-14 DAP Urea
4,750004,20-20 10-26-26 28-28
...,...,...
249995,999995,17-17-17 28-28 20-20
249996,999996,10-26-26 14-35-14 17-17-17
249997,999997,DAP 14-35-14 Urea
249998,999998,10-26-26 17-17-17 28-28


In [31]:
from sklearn.model_selection import StratifiedKFold
from collections import Counter

In [32]:
# 1. Compute class_weights globally (can also be per fold)
counter_full = Counter(y)
max_count_full = max(counter_full.values())
class_weights_full = {cls: max_count_full / count for cls, count in counter_full.items()}

# 2. Stratified CV
kfold = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
fold_accuracies = []
oof_preds = np.zeros((X.shape[0], len(np.unique(y))))

for fold, (train_idx, val_idx) in enumerate(kfold.split(X, y), 1):
    print(f"\n================ Fold {fold} ================")

    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

    # 3. Compute per-instance weights
    counter_fold = Counter(y_tr)
    max_count_fold = max(counter_fold.values())
    sample_weights = y_tr.map(lambda cls: max_count_fold / counter_fold[cls])

    # 4. Instantiate XGBoost model
    XGB_model = xg.XGBClassifier(
        max_depth=12,
        colsample_bytree=0.467,
        subsample=0.86,
        n_estimators=4000,
        learning_rate=0.03,
        gamma=0.26,
        max_delta_step=4,
        reg_alpha=2.7,
        reg_lambda=1.4,
        objective='multi:softprob',
        random_state=13,
        enable_categorical=True,
        tree_method='hist',     
        device='cuda',
        early_stopping_rounds=150,        
    )

    # 5. Fit with early stopping
    XGB_model.fit(
        X_tr,
        y_tr,
        sample_weight=sample_weights,
        eval_set=[(X_va, y_va)],
        verbose=200,
    )

    val_labels = XGB_model.predict(X_va)
    val_probas = XGB_model.predict_proba(X_va)

    oof_preds[val_idx] = val_probas
    acc = accuracy_score(y_va, val_labels)
    fold_accuracies.append(acc)
    print(f"✅ Fold {fold} Accuracy: {acc:.4f}")

# 6. Final CV metrics
print("\n🎯 Mean CV Accuracy:", np.mean(fold_accuracies))
print("📈 Std CV Accuracy:", np.std(fold_accuracies))

# Get Top-3 predicted class indices
top3_preds = np.argsort(oof_preds, axis=1)[:, ::-1][:, :3]


================ Fold 1 ================
[0]	validation_0-mlogloss:1.94566
[200]	validation_0-mlogloss:1.91935
[400]	validation_0-mlogloss:1.90982
[600]	validation_0-mlogloss:1.90566
[800]	validation_0-mlogloss:1.90391
[1000]	validation_0-mlogloss:1.90314
[1200]	validation_0-mlogloss:1.90276
[1400]	validation_0-mlogloss:1.90284
[1423]	validation_0-mlogloss:1.90287
✅ Fold 1 Accuracy: 0.2154

================ Fold 2 ================
[0]	validation_0-mlogloss:1.94569
[200]	validation_0-mlogloss:1.91987
[400]	validation_0-mlogloss:1.91058
[600]	validation_0-mlogloss:1.90659
[800]	validation_0-mlogloss:1.90491
[1000]	validation_0-mlogloss:1.90431
[1200]	validation_0-mlogloss:1.90422
[1243]	validation_0-mlogloss:1.90421
✅ Fold 2 Accuracy: 0.2124

================ Fold 3 ================
[0]	validation_0-mlogloss:1.94564
[200]	validation_0-mlogloss:1.91894
[400]	validation_0-mlogloss:1.90917
[600]	validation_0-mlogloss:1.90455
[800]	validation_0-mlogloss:1.90255
[1000]	validation_0-mlogloss:

KeyboardInterrupt: 

In [ ]:
map3_score = mapk(
    y.values.tolist(), 
    top3_preds.tolist(),  
    k=3
)
print(f"\n📊 Mean Average Precision @3 (MAP@3): {map3_score:.5f}")


📊 Mean Average Precision @3 (MAP@3): 0.35309


In [ ]:
preds1 = model.predict_proba(X_test)
preds2 = XGB_model.predict_proba(X_test)

In [ ]:
ensemble_test_preds = (0.2 * preds2 + 0.8 * preds1)

submission2 = pd.read_csv("sample_submission.csv")

top3_indices = np.argsort(ensemble_test_preds, axis=1)[:, -3:][:, ::-1]

top3_fertilizers = [' '.join(fertilizer_encoder.inverse_transform(row)) for row in top3_indices]

submission2 = pd.DataFrame({
    'ID': test['id'],
    'Fertilizer Name': top3_fertilizers
})

submission2.to_csv('submissionensemble10.csv', index=False)

In [ ]:
submission2.head()

,ID,Fertilizer_Name
0,750000,DAP 10-26-26 28-28
1,750001,17-17-17 20-20 10-26-26
2,750002,20-20 28-28 10-26-26
3,750003,14-35-14 DAP 17-17-17
4,750004,20-20 Urea 10-26-26


In [ ]:
train.head()

,id,Temparature,Humidity,Moisture,Soil_Type,Crop_Type,Nitrogen,Potassium,Phosphorous,Fertilizer_Name
0,0,37,70,36,1,8,36,4,5,4
1,1,27,69,65,4,4,30,6,18,4
2,2,29,63,32,4,4,24,12,16,2
3,3,35,62,54,4,0,39,12,4,0
4,4,35,58,43,3,6,37,2,16,5


In [ ]:
new_train = pd.read_csv('train.csv')
new_test = pd.read_csv('test.csv')

In [ ]:
new_train.head()

,id,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,0,37,70,36,Clayey,Sugarcane,36,4,5,28-28
1,1,27,69,65,Sandy,Millets,30,6,18,28-28
2,2,29,63,32,Sandy,Millets,24,12,16,17-17-17
3,3,35,62,54,Sandy,Barley,39,12,4,10-26-26
4,4,35,58,43,Red,Paddy,37,2,16,DAP


In [ ]:
X = new_train.drop(columns=['id', 'Fertilizer Name'], errors='ignore')
X_test = new_test.drop(columns=['id'], errors='ignore')

y = new_train["Fertilizer Name"]

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, shuffle=True, random_state=SEED)

In [ ]:
from catboost import CatBoostClassifier, Pool

# Define categorical features
cat_features = ['Soil Type', 'Crop Type']  # Add other categorical columns if any

# Create CatBoost Pool with specified categorical features
train_pool = Pool(X_train, y_train, cat_features=cat_features)
val_pool = Pool(X_val, y_val, cat_features=cat_features)

# Initialize and train CatBoost model
cat_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    random_seed=SEED,
    gpu_ram_part=0.95,  # Utilize GPU if available
    task_type="GPU"     # Use GPU acceleration
)

# Fit the model
cat_model.fit(
    train_pool,
    eval_set=val_pool,
    early_stopping_rounds=50,
    verbose=200
)

# Get predictions
probs = cat_model.predict_proba(X_test)
top3_indices = np.argsort(probs, axis=1)[:, -3:][:, ::-1]

0:	learn: 1.9450360	test: 1.9450892	best: 1.9450892 (0)	total: 20.2ms	remaining: 20.2s
200:	learn: 1.9218504	test: 1.9294902	best: 1.9294902 (200)	total: 3.88s	remaining: 15.4s
400:	learn: 1.9116783	test: 1.9254933	best: 1.9254933 (400)	total: 7.71s	remaining: 11.5s
600:	learn: 1.9036608	test: 1.9233704	best: 1.9233704 (600)	total: 11.4s	remaining: 7.54s
800:	learn: 1.8966321	test: 1.9221321	best: 1.9221321 (800)	total: 15.1s	remaining: 3.76s
999:	learn: 1.8897356	test: 1.9212942	best: 1.9212942 (999)	total: 18.8s	remaining: 0us
bestTest = 1.921294167
bestIteration = 999


In [ ]:
preds3 = cat_model.predict_proba(X_test)

ensemble_test_preds = (0.33 * preds2 + 0.33 * preds1 + 0.34 * preds3)

submission3 = pd.read_csv("sample_submission.csv")

top3_indices = np.argsort(ensemble_test_preds, axis=1)[:, -3:][:, ::-1]

top3_fertilizers = [' '.join(fertilizer_encoder.inverse_transform(row)) for row in top3_indices]

submission3 = pd.DataFrame({
    'ID': test['id'],
    'Fertilizer Name': top3_fertilizers
})

submission3.to_csv('submissionensemble_cats_xgb_optuna.csv', index=False)

In [ ]:
submission3

,ID,Fertilizer_Name
0,750000,DAP 10-26-26 28-28
1,750001,17-17-17 20-20 10-26-26
2,750002,20-20 28-28 10-26-26
3,750003,14-35-14 DAP 17-17-17
4,750004,20-20 Urea 10-26-26
...,...,...
249995,999995,Urea 17-17-17 28-28
249996,999996,14-35-14 10-26-26 Urea
249997,999997,DAP Urea 14-35-14
249998,999998,10-26-26 28-28 17-17-17


In [ ]:
print("TODO: submit submission2 and submission3")

TODO: submit submission2 and submission3


In [ ]:
ensemble_test_preds = preds2

submission5 = pd.read_csv("sample_submission.csv")

top3_indices = np.argsort(ensemble_test_preds, axis=1)[:, -3:][:, ::-1]

top3_fertilizers = [' '.join(fertilizer_encoder.inverse_transform(row)) for row in top3_indices]

submission2 = pd.DataFrame({
    'ID': test['id'],
    'Fertilizer Name': top3_fertilizers
})

submission2.to_csv('submisX.csv', index=False)